In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "local_server" : {
        "transport" : "stdio", 
        "command" : "python", 
        "args" : ["resources/2.1_mcp_server.py"]
    }
})

In [19]:
tools = await client.get_tools()

print(tools)

[StructuredTool(name='web_search', description='does web search for information', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'web_searchArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x118f74a40>)]


In [20]:
resources = await client.get_resources("local_server")
print(resources)

[Blob 4764652464]


In [21]:
prompt = await client.get_prompt("local_server","my_prompt")
system_prompt = prompt[0].content
system_prompt

'\n    You are a helpful assistant that answers user questions about LangChain, LangGraph and LangSmith.\n\n    You can use the following tools/resources to answer user questions:\n    - search_web: Search the web for information\n    - github_file: Access the langchain-ai repo files\n\n    If the user asks a question that is not related to LangChain, LangGraph or LangSmith, you should say "I\'m sorry, I can only answer questions about LangChain, LangGraph and LangSmith."\n\n    You may try multiple tool and resource calls to answer the user\'s question.\n\n    You may also ask clarifying questions to the user to better understand their question.\n    '

In [26]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model = "gemini-3.1-flash-lite-preview")

In [27]:
from langchain.agents import create_agent
agent = create_agent(model = model, tools = tools, system_prompt = system_prompt)

In [29]:
from langchain.messages import HumanMessage
response = agent.invoke({"messages": [HumanMessage("Explain langgraph vs langchain in 100 words")]})
response

{'messages': [HumanMessage(content='Explain langgraph vs langchain in 100 words', additional_kwargs={}, response_metadata={}, id='812fd55b-bf00-4b01-97b1-02f27cc411ec'),
  AIMessage(content=[{'type': 'text', 'text': 'LangChain is a framework designed to build LLM-powered applications by chaining together components like prompts, models, and retrievers. It excels at creating linear, directed sequences of tasks.\n\nLangGraph is built on top of LangChain, designed specifically for creating **stateful, multi-actor applications** with LLMs. While LangChain handles sequences, LangGraph introduces cycles and persistence, allowing agents to iterate, loop, and maintain state over long-running processes.\n\nIn short: Use LangChain to build standard LLM pipelines (chains); use LangGraph when you need complex, agentic workflows that require reasoning loops, memory, and multi-step collaboration.', 'extras': {'signature': 'EjQKMgEMOdbHPPo1KUbjjxZa+otgcZu7WPsefN8sQ2zvC1lQP82M3yCLABZM3UQ61PZH8FNV'}}],